In [1]:
import pandas as pd
from pathlib import Path
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import numpy as np

import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

In [2]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .config("spark.python.worker.faulthandler.enabled", "true") # se algum worker crashar de novo, imprime o traceback nativo real
    .master("local[*]")
    .appName("eda")
    .getOrCreate()
)

## Funções

### Agregações de Features x Target

In [3]:
AGG_FUNCS_FEATURES = {
    "avg": F.mean,
    "median": F.median,
    "min": F.min,
    "max": F.max,
    #"var": F.variance,
    "p95": lambda c: F.percentile_approx(c, 0.95),
}

AGG_FUNCS_TARGET = {
    "avg": F.mean,
    "sum": F.sum,
    "max": F.max,
}


def build_feature_target_stats_df(df, features, target, group_cols, consistency_flag="is_tracking_inconsistent"):
    """
    Agrega feature(s) e o target por group_cols num único groupBy().agg().
    As features usam F.when(~consistency_flag, coluna) — o próprio agregador
    (mean/median/min/max/variance/percentile_approx) ignora os NULLs gerados
    pelo when, então eventos com tracking inconsistente simplesmente não
    contribuem pra estatística da feature, sem precisar filtrar o df antes
    (surface_area, stretch_index etc. ficam distorcidas por esses eventos —
    ver sanity_check_features_tracking.ipynb). O target usa a coluna direto,
    sem condicional — não é sensível à mesma inconsistência.

    Parâmetros
    ----------
    df : DataFrame Spark contendo as colunas de features/target, group_cols
         e a coluna consistency_flag.
    features : str ou list[str]
        Nome(s) da(s) coluna(s) de feature a agregar.
    target : str
        Nome da coluna alvo (recebe as agregações de AGG_FUNCS_TARGET).
    group_cols : list[str]
        Colunas de agrupamento (ex: chave de ciclo de posse).
    consistency_flag : str
        Coluna booleana que marca tracking inconsistente (True = inconsistente).

    Retorna
    -------
    df_agg : DataFrame Spark, uma linha por grupo, com as colunas agregadas
             (formato "{agregacao}_{coluna}", ex: "avg_surface_area").
    """
    if isinstance(features, str):
        features = [features]

    agg_exprs = []

    for col_name in features:
        consistent_col = F.when(~F.col(consistency_flag), F.col(col_name))
        for prefix, func in AGG_FUNCS_FEATURES.items():
            alias = f"{prefix}_{col_name}"
            agg_exprs.append(F.round(func(consistent_col), 3).alias(alias))

    for prefix, func in AGG_FUNCS_TARGET.items():
        alias = f"{prefix}_{target}"
        agg_exprs.append(F.round(func(F.col(target)), 3).alias(alias))

    df_agg = df.groupBy(*group_cols).agg(*agg_exprs)

    return df_agg

In [4]:
def plot_correlation_heatmap(df_agg, features, target, target_aggs=None):
    """
    Monta UM heatmap de correlação (Spearman) cruzando TODAS as agregações
    de TODAS as features pedidas (AGG_FUNCS_FEATURES — avg/median/min/max/
    p95, uma coluna por combinação agregação×feature) com a(s) agregação(ões)
    do target pedida(s) em target_aggs. Aceita uma feature (str) ou várias
    (list) — a chamada é sempre a mesma, só muda quantas colunas entram na
    matriz.

    Parâmetros
    ----------
    df_agg : DataFrame Spark ou pandas retornado por build_feature_target_stats_df.
    features : str ou list[str]
        Nome(s) da(s) feature(s) originais (sem prefixo de agregação).
    target : str
        Nome da coluna alvo original (sem prefixo de agregação).
    target_aggs : str ou list[str]
        Chave(s) de AGG_FUNCS_TARGET a usar pro target (default: todas).
    """
    if isinstance(features, str):
        features = [features]
    if target_aggs is None:
        target_aggs = list(AGG_FUNCS_TARGET.keys())
    elif isinstance(target_aggs, str):
        target_aggs = [target_aggs]

    df_agg_pd = df_agg.toPandas() if not isinstance(df_agg, pd.DataFrame) else df_agg

    agg_names = list(AGG_FUNCS_FEATURES.keys())
    feature_cols = [f"{prefix}_{feature}" for feature in features for prefix in agg_names]
    target_cols = [f"{ta}_{target}" for ta in target_aggs]

    corr = df_agg_pd[feature_cols + target_cols].corr(method='spearman')

    n = len(corr.columns)

    fig = go.Figure(
        data=go.Heatmap(
            z=corr.values,
            x=corr.columns,
            y=corr.columns,
            colorscale='RdBu',
            zmin=-1,
            zmax=1,
            text=corr.round(3).values,
            texttemplate="%{text}",
            colorbar=dict(title="Correlação"),
        )
    )

    fig.update_layout(
        title=f"Correlação (Spearman) — agregações das features x {'/'.join(target_aggs)}_{target}",
        template="simple_white",
        height=700,
        width=1000,
        xaxis=dict(tickangle=-45),
    )

    fig.show()

In [5]:
# caminho pra pasta com dados
data_folder_path = Path().resolve().parent.parent / "data"

In [6]:
# base de ameaça criada
threat_dataset_path = str(data_folder_path / "threat_dataset")
features_dataset_path = str(data_folder_path / "features_dataset")

df_threat = spark.read.parquet(threat_dataset_path)
df_features = spark.read.csv(features_dataset_path, header=True, inferSchema=True)

In [7]:
threat_cols = [
    'gameId',
    'competitionId',
    'season',
    'date',
    'eventId',
    'period',
    'startGameClock',
    'startFormattedGameClock',
    'homeTeam',
    'flipped_homeTeam',
    'eventType',
    'eventTypeDescription',
    'eventSubTypeDescription',
    'eventOutcomeDescription',
    'eventPlayerName',
    'eventPlayerPositionType',
    'eventPlayerPositionGroup',
    'eventTeamName',
    'competitionName',
    'homeTeamName',
    'opponentTeamName',
    'possession_id',
    'attackers',
    'defenders',
    'attackingPlayersNorm',
    'defendingPlayersNorm',
    'threat_score',
    'threat_score_impact'
]

In [8]:
df_threat_features = (
    df_threat.select(*threat_cols)
    .join(
        df_features,
        on=["competitionId", "season", "gameId", "eventId"],
        how='inner'
    )
)

# Flag auxiliar: indica se esse evento é o primeiro da sua posse
w_pos = Window.partitionBy("competitionId", "season", "gameId").orderBy("startGameClock")

df_threat_features = df_threat_features.withColumn(
    "first_cycle_event",
    F.lag("possession_id", 1).over(w_pos).isNull() |
    (F.lag("possession_id", 1).over(w_pos) != F.col("possession_id"))
)

df_threat_features.show()

+-------------+---------+------+--------------------+----------+------+--------------+-----------------------+--------+----------------+------------+--------------------+-----------------------+-----------------------+------------------+-----------------------+------------------------+--------------+---------------+--------------+----------------+-------------+---------+---------+--------------------+--------------------+------------+-------------------+------------------------+------------+-------------+-----------+----------+-------------+------------------+--------------------------+-------------------------+------------+------------+------------+-----------------------+-----------------------+-----------------+
|competitionId|   season|gameId|             eventId|      date|period|startGameClock|startFormattedGameClock|homeTeam|flipped_homeTeam|   eventType|eventTypeDescription|eventSubTypeDescription|eventOutcomeDescription|   eventPlayerName|eventPlayerPositionType|eventPlayerPos

## Abordagem 1: Ciclos de posse começando no goleiro e com todos os 11 jogadores do time defendendo atrás da linha da bola

In [9]:
df_cycle_start_gk_keys = (
    df_threat_features
    .filter(
        (F.col('first_cycle_event')) &
        (F.col('eventPlayerPositionType') == 'GK') &
        (F.col('defenders') == 11)
    )
    .select('competitionId', 'season', 'gameId', 'possession_id')
)

df_cycle_start_gk = (
    df_threat_features
    .join(
        F.broadcast(df_cycle_start_gk_keys),
        on=['competitionId', 'season', 'gameId', 'possession_id'],
        how='inner'
    )
)
df_cycle_start_gk = df_cycle_start_gk.cache()
df_cycle_start_gk.show(5)

+-------------+---------+------+-------------+--------------------+----------+------+--------------+-----------------------+--------+----------------+---------+--------------------+-----------------------+-----------------------+----------------+-----------------------+------------------------+-------------+---------------+------------+----------------+---------+---------+--------------------+--------------------+------------+-------------------+------------------------+------------+-------------+-----------+----------+-------------+------------------+--------------------------+-------------------------+------------+------------+------------+-----------------------+-----------------------+-----------------+
|competitionId|   season|gameId|possession_id|             eventId|      date|period|startGameClock|startFormattedGameClock|homeTeam|flipped_homeTeam|eventType|eventTypeDescription|eventSubTypeDescription|eventOutcomeDescription| eventPlayerName|eventPlayerPositionType|eventPlayerPo

In [10]:
qtd_eventos = df_cycle_start_gk.count()
qtd_eventos_total = df_threat_features.count()

print(f'Quantidade de eventos: {qtd_eventos}')
print(f'Quantidade de eventos total na temporada: {qtd_eventos_total}')
print(f'Quantidade de eventos em relação ao total: {qtd_eventos / qtd_eventos_total:.2%}')

Quantidade de eventos: 27820
Quantidade de eventos total na temporada: 448193
Quantidade de eventos em relação ao total: 6.21%


In [11]:
df_possession_agg = (
    df_cycle_start_gk
    .groupBy('competitionId', 'season', 'gameId', 'possession_id')
    .agg(F.countDistinct(F.col('eventId')).alias('qtd_eventos')
    )
    .sort('qtd_eventos', ascending=False)
)

qtd_ciclos = df_possession_agg.count()
qtd_ciclos_total = df_threat_features.select('competitionId', 'season', 'gameId', 'possession_id').distinct().count()

print(f'Quantidade de ciclos: {qtd_ciclos}')
print(f'Quantidade total de ciclos na temporada: {qtd_ciclos_total}')
print(f'Quantidade de ciclos em relação ao total: {qtd_ciclos / qtd_ciclos_total:.2%}')

#df_possession_agg.show()

Quantidade de ciclos: 5784
Quantidade total de ciclos na temporada: 100886
Quantidade de ciclos em relação ao total: 5.73%


In [12]:
(
    df_cycle_start_gk
    .filter((F.col('gameId') == 4807) & (F.col('possession_id') == 198))
).show(100, truncate=True)

+-------------+---------+------+-------------+--------------------+----------+------+--------------+-----------------------+--------+----------------+---------+--------------------+-----------------------+-----------------------+--------------------+-----------------------+------------------------+---------------+---------------+------------+----------------+---------+---------+--------------------+--------------------+------------+-------------------+------------------------+------------+-------------+-----------+----------+-------------+------------------+--------------------------+-------------------------+------------+------------+------------+-----------------------+-----------------------+-----------------+
|competitionId|   season|gameId|possession_id|             eventId|      date|period|startGameClock|startFormattedGameClock|homeTeam|flipped_homeTeam|eventType|eventTypeDescription|eventSubTypeDescription|eventOutcomeDescription|     eventPlayerName|eventPlayerPositionType|eve

In [13]:
df_possession_agg_pd = df_possession_agg.toPandas()

fig = go.Figure()

#cores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig.add_trace(
    go.Box(
        y=df_possession_agg_pd['qtd_eventos'],
        name='',
        marker_color='#1f77b4',
        text=df_possession_agg_pd['possession_id'],
        hovertemplate='%{text}<br>Quantidade de eventos: %{y}<extra></extra>',
        #legendgroup=freq,
        #showlegend=True
    )
)

fig.update_layout(
    title=f'Ciclos de posse das partidas',
    height=700,
    width=600
)

fig.update_yaxes(title_text='Quantidade de eventos')
fig.show()

In [ ]:
features = [
    'surface_area',
    'stretch_index',
    'team_length',
    'team_width',
    'defense_width',
    'height_goal_player',
    'height_goal_team_centroide',
    'height_goal_def_centroide',
    'def_mid_dist',
    'def_atk_dist',
    'atk_mid_dist',
    'numeric_superiority_10m',
    'numeric_superiority_20m',
]
target = 'threat_score'
group_cols = ['competitionId', 'season', 'gameId', 'possession_id']

df_features_target_agg = build_feature_target_stats_df(
    df=df_threat_features,
    features=features,
    target=target,
    group_cols=group_cols,
)

df_features_target_agg_pd = df_features_target_agg.toPandas()

for feature in features:
    plot_correlation_heatmap(df_features_target_agg_pd, feature, target, target_aggs=['max'])

## Abordagem 2: Ciclos de posse começando com todos os 11 jogadores do time defendendo atrás da linha da bola

In [15]:
df_cycle_start_gk_keys = (
    df_threat_features
    .filter(
        (F.col('first_cycle_event')) &
        (F.col('defenders') == 11)
    )
    .select('competitionId', 'season', 'gameId', 'possession_id')
)

df_cycle_start_gk = (
    df_threat_features
    .join(
        F.broadcast(df_cycle_start_gk_keys),
        on=['competitionId', 'season', 'gameId', 'possession_id'],
        how='inner'
    )
)
df_cycle_start_gk = df_cycle_start_gk.cache()
df_cycle_start_gk.show(5)

+-------------+---------+------+-------------+--------------------+----------+------+--------------+-----------------------+--------+----------------+------------+--------------------+-----------------------+-----------------------+---------------+-----------------------+------------------------+-------------+---------------+------------+----------------+---------+---------+--------------------+--------------------+------------+-------------------+------------------------+------------+-------------+-----------+----------+-------------+------------------+--------------------------+-------------------------+------------+------------+------------+-----------------------+-----------------------+-----------------+
|competitionId|   season|gameId|possession_id|             eventId|      date|period|startGameClock|startFormattedGameClock|homeTeam|flipped_homeTeam|   eventType|eventTypeDescription|eventSubTypeDescription|eventOutcomeDescription|eventPlayerName|eventPlayerPositionType|eventPlay

In [16]:
qtd_eventos = df_cycle_start_gk.count()
qtd_eventos_total = df_threat_features.count()

print(f'Quantidade de eventos: {qtd_eventos}')
print(f'Quantidade de eventos total na temporada: {qtd_eventos_total}')
print(f'Quantidade de eventos em relação ao total: {qtd_eventos / qtd_eventos_total:.2%}')

Quantidade de eventos: 129594
Quantidade de eventos total na temporada: 448193
Quantidade de eventos em relação ao total: 28.91%


In [17]:
df_possession_agg = (
    df_cycle_start_gk
    .groupBy('competitionId', 'season', 'gameId', 'possession_id')
    .agg(F.countDistinct(F.col('eventId')).alias('qtd_eventos')
    )
    .sort('qtd_eventos', ascending=False)
)

qtd_ciclos = df_possession_agg.count()
qtd_ciclos_total = df_threat_features.select('competitionId', 'season', 'gameId', 'possession_id').distinct().count()

print(f'Quantidade de ciclos: {qtd_ciclos}')
print(f'Quantidade total de ciclos na temporada: {qtd_ciclos_total}')
print(f'Quantidade de ciclos em relação ao total: {qtd_ciclos / qtd_ciclos_total:.2%}')

#df_possession_agg.show()

Quantidade de ciclos: 25506
Quantidade total de ciclos na temporada: 100886
Quantidade de ciclos em relação ao total: 25.28%


In [18]:
df_possession_agg_pd = df_possession_agg.toPandas()

fig = go.Figure()

#cores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig.add_trace(
    go.Box(
        y=df_possession_agg_pd['qtd_eventos'],
        name='',
        marker_color='#1f77b4',
        text=df_possession_agg_pd['possession_id'],
        hovertemplate='%{text}<br>Quantidade de eventos: %{y}<extra></extra>',
        #legendgroup=freq,
        #showlegend=True
    )
)

fig.update_layout(
    title=f'Ciclos de posse das partidas',
    height=700,
    width=600
)

fig.update_yaxes(title_text='Quantidade de eventos')
fig.show()

In [ ]:
features = [
    'surface_area',
    'stretch_index',
    'team_length',
    'team_width',
    'defense_width',
    'height_goal_player',
    'height_goal_team_centroide',
    'height_goal_def_centroide',
    'def_mid_dist',
    'def_atk_dist',
    'atk_mid_dist',
    'numeric_superiority_10m',
    'numeric_superiority_20m',
]
target = 'threat_score'
group_cols = ['competitionId', 'season', 'gameId', 'possession_id']

df_features_target_agg = build_feature_target_stats_df(
    df=df_threat_features,
    features=features,
    target=target,
    group_cols=group_cols,
)

df_features_target_agg_pd = df_features_target_agg.toPandas()

for feature in features:
    plot_correlation_heatmap(df_features_target_agg_pd, feature, target, target_aggs=['max'])